# SimBank — Pipeline de Principio a Fin

Dataset sintético de banca (100k casos, proceso de préstamos) con intervención `contact_headquarters`.  
Formato: **pickle**. Sin necesidad de pm4py.

## Pipeline completo
```
EventLog (pickle)
  → preprocess_event_log       # clean.parquet
  → encode_prefixes            # prefixes.npz + vocab_activity.json
  → build_mdp_dataset          # D_offline.npz
  → validate_and_split_dataset # splits.json
  → train_tdqn                 # Q_theta.ckpt
  → fit_behavior + doubly_robust_estimate  # ope_dr.json
  → explain_policy             # risk_explanations.json, deltaQ_explanations.json
  → distill_policy             # tree.pkl, rules.sql
```

In [ ]:
from pathlib import Path

import xppm

print("xppm version:", xppm.__version__)

BASE = Path("..")
DATA = BASE / "data"
ART = BASE / "artifacts"
CFG = BASE / "configs" / "config.yaml"
DS = "simbank"

## 1. Definir el schema del event log

SimBank usa `case_nr` como identificador de caso (en vez del estándar `case_id`).  
El outcome ya existe como columna en el log, así que usamos `mode='from_column'`.

In [ ]:
from xppm import EventLogSchema, OutcomeConfig

schema = EventLogSchema(
    case_id="case_nr",  # columna original → se renombra a 'case_id' internamente
    activity="activity",
    timestamp="timestamp",
    outcome=OutcomeConfig(
        mode="from_column",  # el outcome ya existe en el log
        column="outcome",
    ),
)

print("Mapeo de columnas:", schema.column_mapping)

## 2. Preprocesar el event log

In [ ]:
from xppm import preprocess_event_log

raw_path = DATA / "raw" / "loan_log_[_time_contact_HQ_]_100000_train_normal"
clean_path = DATA / DS / "interim" / "clean.parquet"
clean_path.parent.mkdir(parents=True, exist_ok=True)

stats = preprocess_event_log(raw_path, clean_path, schema=schema)

print(f"Casos:          {stats['n_cases']:>10,}")
print(f"Eventos:        {stats['n_events']:>10,}")
print(f"Longitud media: {stats['case_length_mean']:>10.1f} eventos/caso")
print(f"Rango fechas:   {stats['date_min'][:10]} → {stats['date_max'][:10]}")

## 3. Codificar prefijos

Construye un vocabulario de actividades y tokeniza cada traza en secuencias de longitud `max_len=50`.

In [ ]:
from xppm import Config, encode_prefixes

cfg = Config.for_dataset(CFG, DS)

prefixes_path = DATA / DS / "interim" / "prefixes.npz"
vocab_path = DATA / DS / "interim" / "vocab_activity.json"

# El vocab se guarda en la ruta especificada en config encoding.output.vocab_activity_path
cfg.raw["encoding"]["output"]["vocab_activity_path"] = str(vocab_path)

enc_stats = encode_prefixes(
    clean_log_path=clean_path,
    output_path=prefixes_path,
    config=cfg.raw,
)

print(f"Prefijos generados: {enc_stats['n_prefixes']:,}")
print(f"Tamaño vocabulario: {enc_stats['vocab_size']} actividades")
print(f"Guardado en:        {prefixes_path}")

## 4. Construir dataset MDP offline

Genera tuplas `(s, a, r, s', valid_actions)` para cada decisión en el historial.  
La acción `contact_headquarters` es la intervención; `do_nothing` es el no-op.

In [ ]:
from xppm import build_mdp_dataset

mdp_path = DATA / DS / "processed" / "D_offline.npz"
mdp_path.parent.mkdir(parents=True, exist_ok=True)

mdp_stats = build_mdp_dataset(
    prefixes_path=prefixes_path,
    clean_log_path=clean_path,
    vocab_path=vocab_path,
    output_path=mdp_path,
    config=cfg.raw,
)

print(f"Transiciones totales:  {mdp_stats['n_transitions']:>10,}")
print(f"Casos incluidos:       {mdp_stats['n_cases_used']:>10,}")
print(f"Reward medio terminal: {mdp_stats['mean_terminal_reward']:>10.3f}")
print(
    f"Tasa de intervención:  "
    f"{mdp_stats['action_counts'].get(1, 0) / mdp_stats['n_transitions']:.1%}"
)

## 5. Validar y partir el dataset

In [ ]:
from xppm import validate_and_split_dataset

splits_path = DATA / DS / "processed" / "splits.json"

split_stats = validate_and_split_dataset(
    npz_path=mdp_path,
    splits_path=splits_path,
    config=cfg.raw,
)

for split, info in split_stats["splits"].items():
    print(f"  {split:5s}: {info['n_cases']:,} casos, {info['n_transitions']:,} transiciones")

## 6. Entrenar TDQN (Transformer Double Q-Network)

Entrenamiento **offline** sobre el dataset histórico.  
> Para producción usa `max_steps=200_000`. Aquí usamos `5_000` para demo rápido.

In [ ]:
from xppm import TDQNConfig, train_tdqn

ckpt_dir = ART / "models" / "tdqn" / "simbank_demo"
ckpt_dir.mkdir(parents=True, exist_ok=True)

tdqn_cfg = TDQNConfig(
    # Datos
    npz_path=str(mdp_path),
    splits_path=str(splits_path),
    vocab_path=str(vocab_path),
    # Arquitectura Transformer
    max_len=50,
    d_model=128,
    n_heads=4,
    n_layers=3,
    n_actions=2,  # do_nothing | contact_headquarters
    # Hiperparámetros
    batch_size=128,
    learning_rate=3e-4,
    gamma=0.99,
    max_steps=5_000,  # demo: 5k pasos (~3 min en CPU)
    eval_every=1_000,
    save_every=5_000,
    double_dqn=True,
    target_update_every=500,
    grad_clip_norm=10.0,
    device="cuda",  # cambia a 'cpu' si no tienes GPU
    seed=42,
)

result = train_tdqn(tdqn_cfg, checkpoint_dir=ckpt_dir)

ckpt_path = ckpt_dir / "Q_theta.ckpt"
print(f"Checkpoint: {ckpt_path}")
print(f"Loss final:  {result['final_loss']:.4f}")
print(f"Q-mean:      {result['final_q_mean']:.4f}")

## 7. Off-Policy Evaluation (Doubly Robust)

Estima cuánto vale la política TDQN **sin necesidad de desplegarla** en producción.  
Compara π_e (TDQN) contra μ (política histórica) usando el estimador Doubly Robust.

In [ ]:
from xppm import doubly_robust_estimate, fit_behavior_policy_tdqn_encoder

# 7a. Estimar la política de comportamiento histórico π_b(a|s)
behavior = fit_behavior_policy_tdqn_encoder(
    npz_path=mdp_path,
    splits_path=splits_path,
    ckpt_path=ckpt_path,
    vocab_path=vocab_path,
    config=cfg.raw,
)
print(
    f"Behavior model — val_acc: {behavior.metrics['val_acc']:.3f}, "
    f"val_nll: {behavior.metrics['val_nll']:.3f}"
)

# 7b. Estimador DR en split de test
ope = doubly_robust_estimate(
    ckpt_path=ckpt_path,
    dataset_path=mdp_path,
    splits_path=splits_path,
    vocab_path=vocab_path,
    config=cfg.raw,
    behavior=behavior,
    rho_cap=20.0,
    n_bootstrap=200,
)

res = ope["results"]
print("\nPolítica TDQN (π_e):")
print(
    f"  DR estimate: {res['pi_e']['dr_mean']:.4f}  "
    f"95% CI [{res['pi_e']['ci_low']:.4f}, {res['pi_e']['ci_high']:.4f}]"
)
print("\nPolítica histórica (μ):")
print(f"  DR estimate: {res['behavior']['dr_mean']:.4f}")

## 8. Explicar la política (Integrated Gradients)

Genera dos tipos de explicaciones por caso:
- **Risk** (φ^V): ¿qué actividades del prefijo explican el riesgo estimado?
- **ΔQ** (φ^ΔQ): ¿qué explica la decisión de intervenir vs. no hacer nada?

In [ ]:
import json

from xppm import explain_policy

# Apuntar checkpoint y reducir n_cases para demo
cfg.raw["xai"]["checkpoint_path"] = str(ckpt_path)
cfg.raw["xai"]["n_cases"] = 30
cfg.raw["xai"]["seed"] = 42

xai_paths = explain_policy(cfg.raw, config_hash=cfg.config_hash)

print("Artefactos XAI generados:")
for name, path in xai_paths.items():
    print(f"  {name:20s} → {path}")

In [ ]:
# Inspeccionar una explicación de riesgo
risk_data = json.loads(xai_paths["risk"].read_text())
item = risk_data["items"][0]  # primera transición seleccionada

print(f"Caso {item['case_id']}, paso t={item['t']}")
print(f"  Acción recomendada : {item['a_star_name']}")
print(f"  V(s) = Q(s, a*)    : {item['V']:.4f}")
print()
print("  Top actividades que explican el riesgo:")
for tok in item["top_tokens"][:5]:
    bar = "█" * int(tok["importance"] * 20 / (item["top_tokens"][0]["importance"] + 1e-9))
    print(f"    [{tok['position']:2d}] {tok['token_name']:<30s} {bar} {tok['importance']:.4f}")

In [ ]:
# Inspeccionar una explicación de intervención (ΔQ)
dq_data = json.loads(xai_paths["deltaQ"].read_text())
item_dq = dq_data["items"][0]

print(f"Caso {item_dq['case_id']}, paso t={item_dq['t']}")
print(f"  a*        : {item_dq['a_star_name']}")
print(f"  a_contrast: {item_dq['a_contrast_name']}")
print(f"  ΔQ = Q(s,a*) - Q(s,a_contrast) = {item_dq['delta_q']:.4f}")
print()
print("  Actividades que impulsan la intervención:")
for tok in item_dq["top_drivers"][:5]:
    print(f"    [{tok['position']:2d}] {tok['token_name']:<30s} {tok['importance']:.4f}")

## 9. Destilar la política a árbol de decisión (VIPER)

El árbol es completamente interpretable y se puede exportar a SQL para auditoría.

In [ ]:
from xppm import distill_policy

cfg.raw["distill"]["teacher_checkpoint"] = str(ckpt_path)
cfg.raw["distill"]["sample"]["n_states"] = 500  # demo; producción: 2000
cfg.raw["distill"]["surrogate"]["max_depth"] = 4

distill_result = distill_policy(cfg.raw)

print("Fidelidad del árbol vs TDQN:")
print(f"  train: {distill_result['fidelity_train']:.3f}")
print(f"  test:  {distill_result['fidelity_test']:.3f}")
print(f"\nProfundidad del árbol: {distill_result['tree_depth']}")
print(f"Hojas:                 {distill_result['n_leaves']}")

In [ ]:
# Mostrar las primeras reglas del árbol
print("Primeras reglas del árbol de decisión:")
for rule in distill_result.get("rules", [])[:8]:
    print(" ", rule)